In [1]:
!pip install -q langchain-community langchain-google-genai langchain-core faiss-cpu scikit-learn pandas numpy tqdm
!pip install --upgrade langchain
!pip install --upgrade langchain-text-splitters
!pip install --upgrade tqdm
!pip install langchain_huggingface


In [2]:
!pip install faiss-cpu
from langchain_community.vectorstores import FAISS

In [3]:
from google.colab import drive
import pandas as pd
import os
import zipfile

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/RAG Assignment/Dataset.zip'
extract_path = '/content/dataset'

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    if os.path.exists('/content/Dataset'):
        os.rename('/content/Dataset', extract_path)

res_path = '/content/drive/MyDrive/RAG Assignment/RES'
SHEET_ID = "1g1qkvooaqaGk0rqgGTQw2YPltc_CZd1km-g33_vxMCE"
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"

df = pd.read_csv(csv_url)
csv_output = "/content/res.csv"
df.to_csv(csv_output, index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# === Configuration ===
DOCUMENTS_PATH = "/content/export_1"
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
BATCH_SIZE = 1024
OUTPUT_DIR = "faiss_db"

# === Load text documents ===
def load_text_documents(path):
  documents = []
  for filename in os.listdir(path):
      if filename.endswith(".txt"):
          file_path = os.path.join(path, filename)
          loader = TextLoader(file_path, encoding='utf-8')
          raw_docs = loader.load()
          for doc in raw_docs:
              # Normalize content
              content = doc.page_content.replace("\n", " ").lower().strip()
              doc.page_content = content
              documents.append(doc)
  return documents

print("Loading text documents...")
raw_documents = load_text_documents(DOCUMENTS_PATH)
print(f"Total documents found: {len(raw_documents)}")

subset_idx = len(raw_documents)
raw_documents = raw_documents[:subset_idx]
print(f"Documents selected for embedding: {len(raw_documents)}")

# === Split documents into smaller chunks ===
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=50,
    length_function=len
)

texts = []
for doc in tqdm(raw_documents, desc="Splitting documents into chunks"):
    chunks = text_splitter.split_documents([doc])
    texts.extend(chunks)

print(f"Total chunks created: {len(texts)}")

# === Initialize embeddings ===
print(f"Loading embedding model: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda"}
)

# === Build FAISS vectorstore ===
vectorstore = None

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding and adding to FAISS"):
    batch = texts[i:i + BATCH_SIZE]
    batch_embeddings = embeddings.embed_documents([t.page_content for t in batch])

    if vectorstore is None:
        vectorstore = FAISS.from_texts(
            [t.page_content for t in batch],
            embeddings
        )
    else:
        vectorstore.add_texts(
            [t.page_content for t in batch],
            embedding=batch_embeddings
        )

# === Save FAISS index ===
os.makedirs(OUTPUT_DIR, exist_ok=True)
vectorstore.save_local(OUTPUT_DIR)
print(f"FAISS vectorstore successfully saved to '{OUTPUT_DIR}'")

Loading text documents...
Total documents found: 16440
Documents selected for embedding: 16440


Splitting documents into chunks: 100%|██████████| 16440/16440 [02:29<00:00, 109.91it/s]
/tmp/ipython-input-888713680.py:53: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Total chunks created: 176506
Loading embedding model: keepitreal/vietnamese-sbert


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Embedding and adding to FAISS: 100%|██████████| 173/173 [1:43:58<00:00, 36.06s/it]


FAISS vectorstore successfully saved to 'faiss_db'


In [ ]:
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# === Configuration ===
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
VECTORSTORE_PATH = "faiss_db"

# === Load FAISS vectorstore ===
print("Loading FAISS vectorstore...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)
print("FAISS vectorstore loaded successfully.")

# === Display a sample record ===
keys = list(vectorstore.docstore._dict.keys())
if not keys:
    raise ValueError("The FAISS docstore is empty. No records found.")

first_key = keys[0]
record = vectorstore.docstore._dict[first_key]

print("\n--- First Record ---")
print(f"ID: {first_key}")
print(f"Content (truncated): {record.page_content[:200]}...")
print(f"Metadata: {record.metadata}")

# === Retrieve the corresponding vector ===
index_id = 0  # first vector
vector = vectorstore.index.reconstruct(index_id)

print("\n--- Corresponding Vector ---")
print(f"Vector size: {len(vector)}")
print(f"First 5 values: {vector[:5]}")


Loading FAISS vectorstore...
FAISS vectorstore loaded successfully.

--- First Record ---
ID: 8dd217cd-ee0d-4592-8ef1-d4bdc33cf1b2
Content (truncated): ủy ban nhân dân   tỉnh kon tum   -------  cộng hòa xã hội  chủ nghĩa việt nam   độc lập - tự do - hạnh phúc    ---------------  số: 3809/kh-ubnd  kon tum, ngày 23  tháng 10 năm 2024     kế hoạch  triể...
Metadata: {}

--- Corresponding Vector ---
Vector size: 768
First 5 values: [ 0.24020351 -0.10855713 -0.00235908  0.01199178  0.0657928 ]


In [ ]:
import os
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.metrics.pairwise import cosine_similarity

# ==================== LOAD DATA ====================
df = pd.read_csv("res.csv")
df = df.head(30)
print(f"Total number of questions: {len(df)}")

# # ==================== LOAD FAISS + EMBEDDINGS ====================
# embedding_model_name = "keepitreal/vietnamese-sbert"
# embeddings = HuggingFaceEmbeddings(
#     model_name=embedding_model_name,
#     model_kwargs={"device": "cpu"}  # or "cuda"
# )

vectorstore = FAISS.load_local(
    "faiss_db",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)

# ==================== LOAD LLM ====================
llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen1.5-1.8B", #Choose your model
    task="text-generation",
    device=0,  # GPU 0
    model_kwargs={"temperature": 0, "max_length": 1024}
)

# ==================== HELPER FUNCTIONS ====================
def normalize_text(text: str) -> str:
    text = str(text).lower().replace("\n", " ").strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def rag_query(question: str, k: int = 5) -> str:
    docs = vectorstore.similarity_search(question, k=k)
    context_texts = []
    for doc in docs:
        if hasattr(doc, "page_content"):
            context_texts.append(doc.page_content)
        elif isinstance(doc, dict) and "page_content" in doc:
            context_texts.append(doc["page_content"])
        else:
            context_texts.append(str(doc))
    context = "\n".join(context_texts)

    prompt = f"""
Bạn là một trợ lý thông minh. Trả lời câu hỏi dựa trên ngữ cảnh sau.
Nếu không đủ thông tin, hãy nói rõ điều đó, không được tự bịa đặt.

Ngữ cảnh:
{context}

Câu hỏi: {question}
Trả lời ngắn gọn và chính xác nhất có thể:
"""

    output = llm.generate([prompt],  max_new_tokens=256)
    generated_text = output.generations[0][0].text
    print(generated_text)
    return generated_text

def cosine_similarity_score(pred: str, true: str, emb_model) -> float:
    if not pred or not true:
        return 0.0
    pred_vec = emb_model.embed_query(pred)
    true_vec = emb_model.embed_query(true)
    return float(cosine_similarity([pred_vec], [true_vec])[0][0])

def jaccard_similarity(pred: str, true: str) -> float:
    pred_words = set(normalize_text(pred).split())
    true_words = set(normalize_text(true).split())
    return len(pred_words & true_words) / len(pred_words | true_words) if pred_words and true_words else 0.0

def token_overlap_score(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not true_words:
        return 0.0
    return sum((Counter(pred_words) & Counter(true_words)).values()) / len(true_words)

def bleu_score_simple(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not pred_words or not true_words:
        return 0.0
    overlap = sum((Counter(pred_words) & Counter(true_words)).values())
    precision = overlap / len(pred_words)
    bp = 1.0 if len(pred_words) >= len(true_words) else np.exp(1 - len(true_words)/len(pred_words))
    return bp * precision

def rouge_l_score(pred: str, true: str) -> float:
    pred_words = normalize_text(pred).split()
    true_words = normalize_text(true).split()
    if not pred_words or not true_words:
        return 0.0
    m, n = len(pred_words), len(true_words)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if pred_words[i-1] == true_words[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    precision = lcs / m
    recall = lcs / n
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

def semantic_containment(pred: str, true: str) -> float:
    pred_norm = normalize_text(pred)
    true_norm = normalize_text(true)
    if not true_norm:
        return 0.0
    if true_norm in pred_norm:
        return 1.0
    pred_words = set(pred_norm.split())
    true_words = set(true_norm.split())
    return 1.0 if true_words.issubset(pred_words) else 0.0

# ==================== EVALUATION ====================
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating RAG"):
    question = row["Câu Hỏi"]
    ground_truth = str(row["Trả lời"]).strip()

    try:
        response = rag_query(question)
    except Exception as e:
        print(f"Error at question {idx}: {e}")
        response = ""

    metrics = {
        "Cosine_Similarity": cosine_similarity_score(response, ground_truth, embeddings),
        "Jaccard_Similarity": jaccard_similarity(response, ground_truth),
        "Token_Overlap": token_overlap_score(response, ground_truth),
        "BLEU_Score": bleu_score_simple(response, ground_truth),
        "ROUGE_L": rouge_l_score(response, ground_truth)
    }

    results.append(metrics)

results_df = pd.DataFrame(results)
mean_metrics = results_df.mean().to_dict()

print("\n=== FINAL RAG EVALUATION SUMMARY ===")
for k, v in mean_metrics.items():
    print(f"{k}: {v:.4f}")

results_df.to_csv("rag_baseline_metrics.csv", index=False)
print("Metrics saved to rag_baseline_metrics.csv")
